# Step 4 - Classification evaluation
Evaluate the trained model on the train and test sets: accuracy, precision, recall, F1-score and confusion matrix.

## Import libraries

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

In [ ]:
SPLIT_DIR = Path("../data/classification/processed")
MODEL_DIR = Path("../models/classification")

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

## Load the saved model

In [ ]:
model = tf.keras.models.load_model(MODEL_DIR / "googlenet_classifier.keras")

with open(MODEL_DIR / "class_names.json") as file:
    class_names = json.load(file)

print(class_names)

## Load the train and test sets
Same preprocessing as in training. No shuffle, so the labels stay in the same order as the predictions.

In [ ]:
def preprocess(images, labels):
    images = tf.cast(images, tf.float32)
    images = tf.keras.applications.inception_v3.preprocess_input(images)
    return images, labels


def load_dataset(folder):
    dataset = tf.keras.utils.image_dataset_from_directory(
        folder,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )
    file_paths = dataset.file_paths
    dataset = dataset.map(preprocess).prefetch(tf.data.AUTOTUNE)
    return dataset, file_paths

In [ ]:
train_ds, train_files = load_dataset(SPLIT_DIR / "train")
test_ds, test_files = load_dataset(SPLIT_DIR / "test")

## Predictions

In [ ]:
def get_predictions(model, dataset):
    y_true = np.concatenate([labels.numpy() for images, labels in dataset])
    probabilities = model.predict(dataset)
    y_pred = np.argmax(probabilities, axis=1)
    return y_true, y_pred

In [ ]:
train_true, train_pred = get_predictions(model, train_ds)
test_true, test_pred = get_predictions(model, test_ds)

## Metrics: accuracy, precision, recall, F1-score
`macro` = the average over the 4 classes, each class counts the same.

In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro"),
        "recall": recall_score(y_true, y_pred, average="macro"),
        "f1_score": f1_score(y_true, y_pred, average="macro"),
    }

In [ ]:
train_metrics = compute_metrics(train_true, train_pred)
test_metrics = compute_metrics(test_true, test_pred)

results = pd.DataFrame({"train": train_metrics, "test": test_metrics})
results.round(4)

## Results for each class (test set)

In [ ]:
print(classification_report(test_true, test_pred, target_names=class_names, digits=4))

## Confusion matrix
Rows = real class, columns = predicted class. The diagonal is the correct predictions.

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title):
    matrix = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Real")
    plt.show()

In [ ]:
plot_confusion_matrix(train_true, train_pred, "Confusion matrix - train")

In [ ]:
plot_confusion_matrix(test_true, test_pred, "Confusion matrix - test")

## Train vs test
If train is much better than test, the model is overfitting.

In [ ]:
results.plot(kind="bar", figsize=(8, 4), ylim=(0, 1), rot=0)
plt.title("Train vs test")
plt.show()

In [ ]:
gap = train_metrics["accuracy"] - test_metrics["accuracy"]
print(f"Accuracy gap train - test: {gap:.4f}")